# 01 · Raw data → S3 data lake → Athena catalog
**AAI-540 · Group 4 · Criteo Display Advertising CTR Prediction**

| Step | Output |
|---|---|
| 1. Acquire the Criteo Kaggle dataset | `train.txt` (45,840,617 labelled rows) on the notebook's EBS volume |
| 2. Land raw data in S3 (immutable raw zone) | `s3://<bucket>/criteo-ctr/raw/train/train.txt` |
| 3. Build a chronological systematic sample | Parquet, partitioned by `day_index`, in the curated zone |
| 4. Catalog both in Athena | `criteo_ctr_db.raw_train`, `criteo_ctr_db.curated_sample` |
| 5. Validate with SQL and write a data-lake manifest | `artifacts/data_lake_manifest.json` |

**Before you run:** notebook instance with **≥ 50 GB** volume (archive + extracted file ≈ 20 GB) and
**≥ 16 GB RAM** (`ml.m5.xlarge` or larger). Uses only `boto3` + the SageMaker SDK; no extra Athena libraries.

**Cost note:** each full-table Athena query on the raw TSV scans ≈ 11 GB (≈ $0.05 at $5/TB). The curated
Parquet sample is what EDA and feature engineering use day-to-day.

In [ ]:
import os, sys, json, time, datetime as dt
sys.path.insert(0, os.path.abspath("../src"))

import boto3
import sagemaker
import pandas as pd
from botocore.exceptions import ClientError

from criteo_ctr import config as C
from criteo_ctr import features as F
from criteo_ctr.athena import Athena
from criteo_ctr.io_utils import Store, s3_uri, lower_columns, canonical_columns, default_data_dir

sess = sagemaker.Session()
boto_sess = sess.boto_session
region = sess.boto_region_name
role = sagemaker.get_execution_role()
bucket = C.BUCKET or sess.default_bucket()
store = Store(bucket, boto_sess)
athena = Athena(boto_sess, s3_uri(bucket, C.ATHENA_RESULTS_PREFIX) + "/", database=C.ATHENA_DATABASE)

print(f"region={region}\nbucket={bucket}\nprefix={C.PREFIX}\nathena db={C.ATHENA_DATABASE}")

## 1 · Acquire the dataset
Two options. **A** pulls from Kaggle with the API (needs `~/.kaggle/kaggle.json` and the competition rules
accepted on kaggle.com). **B** uses an archive you downloaded yourself and uploaded to
`s3://<bucket>/criteo-ctr/landing/`. Both end with `train.txt` on local disk.

The Kaggle `test.txt` has **no labels**, so it is archived in the raw zone but never used for training or
evaluation — our test split is carved out of `train.txt` in notebook 04.

In [ ]:
import glob, shutil, subprocess, tarfile, zipfile

DATA_DIR = default_data_dir()
os.makedirs(DATA_DIR, exist_ok=True)
free_gb = shutil.disk_usage(DATA_DIR).free / 1e9
print(f"scratch dir: {DATA_DIR}  (free: {free_gb:.1f} GB)")
if free_gb < 30:
    print("WARNING: less than 30 GB free - stop the instance and raise its volume size to >= 50 GB.")


def find_file(folder, name):
    hits = glob.glob(os.path.join(folder, "**", name), recursive=True)
    return hits[0] if hits else None


def extract_archives(folder):
    for path in glob.glob(os.path.join(folder, "*")):
        if path.endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                z.extractall(folder)
        elif path.endswith((".tar.gz", ".tgz", ".tar")):
            with tarfile.open(path) as t:
                try:
                    t.extractall(folder, filter="data")   # Python >= 3.12 safe extraction
                except TypeError:
                    t.extractall(folder)

In [ ]:
# ---- Option A: Kaggle API -------------------------------------------------
if not find_file(DATA_DIR, "train.txt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    kaggle_bin = shutil.which("kaggle") or os.path.join(os.path.dirname(sys.executable), "kaggle")
    r = subprocess.run([kaggle_bin, "competitions", "download", "-c", "criteo-display-ad-challenge",
                        "-p", DATA_DIR], capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
    if r.returncode != 0:
        print("Kaggle download did not succeed - use Option B in the next cell.")

In [ ]:
# ---- Option B: archive you uploaded to s3://<bucket>/criteo-ctr/landing/ ---
LANDING_PREFIX = f"{C.PREFIX}/landing/"
if not find_file(DATA_DIR, "train.txt"):
    for key in store.list_keys(LANDING_PREFIX):
        dest = os.path.join(DATA_DIR, os.path.basename(key))
        if not os.path.exists(dest):
            print("downloading", key)
            store.s3.download_file(bucket, key, dest)

extract_archives(DATA_DIR)
LOCAL_TRAIN = find_file(DATA_DIR, "train.txt")
LOCAL_TEST = find_file(DATA_DIR, "test.txt")
assert LOCAL_TRAIN, "train.txt not found - complete Option A or Option B first"
print("train:", LOCAL_TRAIN, f"{os.path.getsize(LOCAL_TRAIN)/1e9:.2f} GB")
print("test :", LOCAL_TEST)

In [ ]:
# Row count drives day_index / event_time, so measure it rather than assume it.
TOTAL_ROWS = int(subprocess.run(["wc", "-l", LOCAL_TRAIN], capture_output=True, text=True).stdout.split()[0])
EXPECTED_ROWS = 45_840_617   # published size of the Kaggle train.txt
print(f"train.txt rows: {TOTAL_ROWS:,}")
if TOTAL_ROWS != EXPECTED_ROWS:
    print(f"NOTE: expected {EXPECTED_ROWS:,}. Check the download completed before continuing.")

with open(LOCAL_TRAIN) as fh:
    first = fh.readline().rstrip("\n").split("\t")
assert len(first) == len(C.RAW_COLUMNS), f"expected {len(C.RAW_COLUMNS)} tab-separated fields, got {len(first)}"
print("first row fields:", len(first), "| label =", first[0])

## 2 · Land the raw data in S3
The raw zone is **write-once**: the original file is stored exactly as downloaded so any model version can be
traced back to its exact input. Everything downstream is derived and can be regenerated.

In [ ]:
t0 = time.time()
RAW_TRAIN_KEY = f"{C.RAW_TRAIN_PREFIX}/train.txt"
existing = store.list_keys(RAW_TRAIN_KEY)
if existing:
    print("raw train already in S3 - not overwriting (raw zone is immutable):", s3_uri(bucket, RAW_TRAIN_KEY))
else:
    store.upload_file(LOCAL_TRAIN, RAW_TRAIN_KEY)
    print(f"uploaded train.txt in {time.time()-t0:.0f}s ->", s3_uri(bucket, RAW_TRAIN_KEY))

if LOCAL_TEST and not store.list_keys(f"{C.RAW_TEST_PREFIX}/test.txt"):
    store.upload_file(LOCAL_TEST, f"{C.RAW_TEST_PREFIX}/test.txt")
    print("archived unlabeled test.txt ->", s3_uri(bucket, f"{C.RAW_TEST_PREFIX}/test.txt"))

head = store.s3.head_object(Bucket=bucket, Key=RAW_TRAIN_KEY)
print(f"S3 object size: {head['ContentLength']/1e9:.2f} GB | encryption: {head.get('ServerSideEncryption')}")

## 3 · Chronological systematic sample → curated zone (Parquet)
Criteo documents `train.txt` as **chronologically ordered** across 7 days, but ships no timestamp. We keep
every *k*-th row (default *k* = 45 → ≈ 1.02 M rows) so the sample spans all 7 days *in order*.
A `head()` sample would cover only part of day 1 and make a chronological split impossible.

`record_id` = row position in the original file (full traceability). `day_index` and `event_time` are derived
from that position and are an **approximation** (traffic is not uniform through a day).

In [ ]:
t0 = time.time()
sample = F.fast_systematic_sample(LOCAL_TRAIN, every=C.SAMPLE_EVERY, total_rows=TOTAL_ROWS)
sample = F.add_record_keys(sample, total_rows=TOTAL_ROWS)
print(f"sampled {len(sample):,} rows (1 in {C.SAMPLE_EVERY}) in {time.time()-t0:.0f}s; "
      f"memory {sample.memory_usage(deep=True).sum()/1e9:.2f} GB")
print("rows per day_index:", sample[C.DAY_INDEX].value_counts().sort_index().to_dict())
sample.head()

In [ ]:
# Write one Parquet file per day_index (Hive-style partitions) with lower-case column
# names, which is what Athena/Glue expect.
store.delete_prefix(C.SAMPLE_PREFIX + "/")
for d, part in sample.groupby(C.DAY_INDEX):
    out = lower_columns(part.drop(columns=[C.DAY_INDEX]))
    out["label"] = out["label"].astype("int32")
    store.put_parquet(out, f"{C.SAMPLE_PREFIX}/day_index={int(d)}/part-00000.parquet")
print("curated sample ->", s3_uri(bucket, C.SAMPLE_PREFIX + "/"))
for k in store.list_keys(C.SAMPLE_PREFIX):
    print("  ", k)

## 4 · Catalog in Athena
Two external tables in `criteo_ctr_db`: the raw TSV (source of truth, full 45.8 M rows) and the curated
Parquet sample (cheap to query, partitioned by day). Tables are dropped and recreated so reruns are idempotent;
dropping an *external* table never deletes the S3 data.

In [ ]:
athena.execute(f"CREATE DATABASE IF NOT EXISTS {C.ATHENA_DATABASE} "
               f"COMMENT 'AAI-540 Group 4 - Criteo CTR data lake'", database="default")

num_cols = ",\n  ".join(f"{c.lower()} bigint" for c in C.NUM_COLS)
cat_cols = ",\n  ".join(f"{c.lower()} string" for c in C.CAT_COLS)

athena.execute(f"DROP TABLE IF EXISTS {C.RAW_TABLE}")
athena.execute(f"""
CREATE EXTERNAL TABLE {C.RAW_TABLE} (
  label int,
  {num_cols},
  {cat_cols}
)
COMMENT 'Criteo Display Advertising Challenge train.txt - raw, immutable, chronological'
ROW FORMAT DELIMITED FIELDS TERMINATED BY '\\t' LINES TERMINATED BY '\\n'
STORED AS TEXTFILE
LOCATION '{s3_uri(bucket, C.RAW_TRAIN_PREFIX)}/'
TBLPROPERTIES ('serialization.null.format'='', 'classification'='csv')
""")

athena.execute(f"DROP TABLE IF EXISTS {C.SAMPLE_TABLE}")
athena.execute(f"""
CREATE EXTERNAL TABLE {C.SAMPLE_TABLE} (
  record_id bigint,
  label int,
  {num_cols},
  {cat_cols},
  event_time double
)
COMMENT '1-in-{C.SAMPLE_EVERY} chronological systematic sample of raw_train'
PARTITIONED BY (day_index int)
STORED AS PARQUET
LOCATION '{s3_uri(bucket, C.SAMPLE_PREFIX)}/'
""")
athena.execute(f"MSCK REPAIR TABLE {C.SAMPLE_TABLE}")
athena.query("SHOW TABLES")

## 5 · Validate the lake with SQL

In [ ]:
qid = athena.execute(f"SELECT 1 FROM {C.RAW_TABLE} LIMIT 1")   # warm-up / sanity
raw_stats = athena.query(f"SELECT count(*) AS n_rows, avg(label) AS ctr, "
                         f"count_if(label NOT IN (0,1)) AS bad_labels FROM {C.RAW_TABLE}")
raw_stats

In [ ]:
sample_by_day = athena.query(f"""
SELECT day_index, count(*) AS n_rows, avg(label) AS ctr, min(record_id) AS first_record, max(record_id) AS last_record
FROM {C.SAMPLE_TABLE} GROUP BY day_index ORDER BY day_index""")
sample_by_day

In [ ]:
n_raw = int(raw_stats.n_rows[0]);  ctr_raw = float(raw_stats.ctr[0])
n_smp = int(sample_by_day.n_rows.sum())
ctr_smp = float((sample_by_day.n_rows * sample_by_day.ctr).sum() / n_smp)

checks = {
    "athena_raw_rows_match_file": n_raw == TOTAL_ROWS,
    "athena_sample_rows_match_dataframe": n_smp == len(sample),
    "labels_are_binary": int(raw_stats.bad_labels[0]) == 0,
    "all_7_days_present": sample_by_day.day_index.tolist() == list(range(7)),
    "sample_ctr_within_0.5pp_of_full": abs(ctr_smp - ctr_raw) < 0.005,
}
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
print(f"\nfull-data CTR {ctr_raw:.4%} | sample CTR {ctr_smp:.4%} | negatives per positive {(1-ctr_raw)/ctr_raw:.2f}")
assert all(checks.values()), "data lake validation failed"

In [ ]:
manifest = {
    "created_utc": dt.datetime.utcnow().isoformat() + "Z",
    "source": "Kaggle - Criteo Display Advertising Challenge (criteo-display-ad-challenge)",
    "raw_train_uri": s3_uri(bucket, RAW_TRAIN_KEY),
    "raw_rows": n_raw, "raw_ctr": ctr_raw,
    "sample_uri": s3_uri(bucket, C.SAMPLE_PREFIX) + "/",
    "sample_every": C.SAMPLE_EVERY, "sample_rows": n_smp, "sample_ctr": ctr_smp,
    "athena_database": C.ATHENA_DATABASE, "athena_tables": [C.RAW_TABLE, C.SAMPLE_TABLE],
    "event_time_note": "synthetic: uniform over 7 days by row position; anchor date is nominal",
    "validation": checks,
}
print(store.put_json(manifest, f"{C.ARTIFACTS_PREFIX}/data_lake_manifest.json"))
manifest